# Prepare the current 23-home cohort

Rebuild the manuscript's hourly cohort and daily means from the saved calibrated measurements.

**Step:** 03. Build the 23-home hourly and daily data.

**Article scope:** the main article and Supplementary Material.

**Inputs:** Saved calibrated hourly data; 31 calibration intervals; the sensor register and home crosswalk.

**Outputs:** Hourly and daily cohort CSV files, monitor screening, and a preparation summary.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Select calibrated measurements

Step 02's calibrated output takes precedence over a supplied calibrated file in `data/`. `INDOOR_AQ_CALIBRATED_INPUT` can select a specific input explicitly.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

# Leave this unset to use the saved calibrated manuscript input.
CALIBRATED_INPUT = os.environ.get('INDOOR_AQ_CALIBRATED_INPUT') or None
# Example: CALIBRATED_INPUT = r'C:\path\to\indoor_all_long_hourly_calibrated.csv'

## Import the required libraries

These imports belong to the active notebook kernel. Install `requirements.txt` in that same environment if an import fails.

In [ ]:
import shutil
import numpy as np
import pandas as pd
from shared import CODE, DATA, crosswalk, output_root, dump_json, data_file, repository_path

## Read calibrated observations and establish the analysis period

Parse timestamps in UTC, retain 2024 and 2025, and reject duplicate sensor-variable-hour keys. Start with the saved calibration output unless `CALIBRATED_INPUT` explicitly identifies another file.

In [ ]:
out = output_root(OUTPUT)

# Saved calibrated hourly measurements (long format: sensor_id, variable, time, value).
source_path = repository_path(CALIBRATED_INPUT) if CALIBRATED_INPUT else data_file('indoor_all_long_hourly_calibrated.csv', out)
x = pd.read_csv(Path(source_path))
x['time'] = pd.to_datetime(x.time, utc=True)

# Keep the two study years, 2024 and 2025 (UTC).
in_study_period = x.time.ge(pd.Timestamp('2024-01-01', tz='UTC')) & x.time.lt(pd.Timestamp('2026-01-01', tz='UTC'))
x = x[in_study_period].copy()
assert not x.duplicated(['sensor_id', 'variable', 'time']).any(), 'Duplicate sensor-variable-hour records'

## Exclude calibration and relocation records

Apply all 31 audited calibration intervals with inclusive boundaries and confirm each interval's observed-hour count. Remove sensor 60 records from December 1, 2025 onward because the sensor moved. These exclusions precede residence screening.

In [ ]:
rules = pd.read_csv(DATA / 'calibration_intervals.csv')
assert len(rules) == 31

removed = pd.Series(False, index=x.index)
exclusions = []
for r in rules.itertuples():
    # Inclusive interval. Intervals must not overlap, and the number of
    # observed hours must equal the audited count.
    in_interval = x.sensor_id.eq(r.sensor) & x.time.between(pd.Timestamp(r.start), pd.Timestamp(r.end))
    assert not (removed & in_interval).any()
    assert x.loc[in_interval & x.value.notna(), 'time'].nunique() == r.prior_audit_hours
    removed |= in_interval
    exclusions.append(dict(sensor=r.sensor, reason='calibration',
                           valid_values=int(x.loc[in_interval, 'value'].count())))

# Sensor 60 was moved to another location on 1 December 2025.
relocation = x.sensor_id.eq(60) & x.time.ge(pd.Timestamp('2025-12-01', tz='UTC'))
removed |= relocation
x = x.loc[~removed].copy()

## Screen the 34 registered monitors

Coverage uses 17,544 potential hours across the two calendar years. A monitor must have coverage strictly greater than 35% and pooled CO2/PM2.5 stagnation strictly below 3%. Stagnation uses runs of at least three identical values after rounding to three decimals. Expect 25 eligible monitors.

In [ ]:
POTENTIAL_HOURS = 17544     # hours in 2024 and 2025 (8,784 + 8,760)
MIN_COVERAGE_PCT = 35       # coverage must be strictly greater than this
MAX_STAGNATION_PCT = 3      # stagnation must be strictly smaller than this
MIN_STUCK_RUN = 3           # identical consecutive values counted as "stuck"

registered = pd.read_csv(DATA / 'registered_sensors.csv').sensor_id.tolist()
screen = []
for sensor in registered:
    g = x[x.sensor_id.eq(sensor)]
    valid_hours = g.loc[g.value.notna(), 'time'].nunique()

    # Stagnation: share of CO2 and PM2.5 values inside a run of at least three
    # identical consecutive values (after rounding to three decimals).
    stuck = denom = 0
    for v in ['co2', 'pm25']:
        s = g.loc[g.variable.eq(v)].sort_values('time').value.dropna().round(3)
        run_length = s.groupby(s.ne(s.shift()).cumsum()).transform('size')
        stuck += int(run_length.ge(MIN_STUCK_RUN).sum())
        denom += len(s)

    coverage = 100 * valid_hours / POTENTIAL_HOURS
    stagnation = 100 * stuck / denom if denom else np.nan
    screen.append(dict(sensor_id=sensor, valid_hours=valid_hours, coverage_pct=coverage,
                       stagnation_pct=stagnation,
                       passes=coverage > MIN_COVERAGE_PCT and stagnation < MAX_STAGNATION_PCT))

screen = pd.DataFrame(screen)
assert len(screen) == 34 and screen.passes.sum() == 25

## Resolve duplicate monitors at the residence level

Two registered sensor pairs share addresses. Rank by usable hours, then by smaller sensor ID for a tie, and retain one monitor per residence. Require exact agreement with the current 23-home crosswalk.

In [ ]:
eligible = screen[screen.passes].copy()

# The register confirms that two sensor pairs share an address. Keep one
# monitor per residence: most usable hours first, then the smaller sensor ID.
duplicate_residences = {75: 'pair_75_91', 91: 'pair_75_91', 90: 'pair_90_95', 95: 'pair_90_95'}
eligible['residence'] = eligible.sensor_id.map(lambda s: duplicate_residences.get(s, f'sensor_{s}'))
selected = (eligible.sort_values(['valid_hours', 'sensor_id'], ascending=[False, True])
            .drop_duplicates('residence'))

cw = crosswalk()
assert set(selected.sensor_id) == set(cw.sensor_id), 'Cohort differs from current manuscript; inspect the input changes.'
screen['retained'] = screen.sensor_id.isin(selected.sensor_id)
screen.to_csv(out / 'tables/monitor_screening.csv', index=False)

## Aggregate valid hours into local calendar days

Convert UTC timestamps to Europe/London before forming dates. A daily mean requires at least 18 nonmissing hourly values, including 23-hour and 25-hour daylight saving days. Keep rows with insufficient hours but set their daily outcome to missing.

In [ ]:
MIN_VALID_HOURS = 18   # valid hours needed for a daily mean

kept = x[x.sensor_id.isin(selected.sensor_id)].copy()
kept = kept.merge(cw[['sensor_id', 'home', 'source_home']], on='sensor_id', validate='many_to_one')
# Local calendar day in Europe/London (23- and 25-hour days at the clock changes).
kept['date'] = kept.time.dt.tz_convert('Europe/London').dt.strftime('%Y-%m-%d')

daily = (kept.groupby(['sensor_id', 'variable', 'date'])
         .value.agg(n_hours='count', y='mean')
         .reset_index())
daily.loc[daily.n_hours.lt(MIN_VALID_HOURS), 'y'] = np.nan   # keep the row, drop the mean
daily = daily.merge(cw[['sensor_id', 'home', 'source_home']], on='sensor_id', validate='many_to_one')
daily['sensor'] = daily.sensor_id
daily['in_primary'] = True
daily.to_csv(out / 'data/daily_home.csv', index=False)

## Save the reproducible inputs

Write the retained hourly series and the daily table to the separate output directory. Copy only the small event and home-ID configuration files required by the model stage.

In [ ]:
# Hourly series of the 23 retained monitors (input to Figures 1-2 and Table S1).
hourly = kept.rename(columns={'sensor_id': 'sensor'})[['sensor', 'home', 'time', 'variable', 'value']]
hourly.to_csv(out / 'data/hourly_primary_23.csv.gz', index=False, compression={'method': 'gzip', 'mtime': 0})

# Small configuration files needed by the model step (08_fit_did_cits_models.R).
for name in ['events.csv', 'home_id_crosswalk_23.csv']:
    shutil.copy2(DATA / name, out / 'data' / name)

## Save the preparation summary

Record cohort sizes, exclusions, and daily aggregation settings for this run.

In [ ]:
dump_json(out / 'logs/preparation_checks.json', dict(
    sensors_screened=34, coverage_eligible=25, homes=23,
    rows_hourly=len(hourly), rows_daily=len(daily), daily_prepared=True,
    calibration_intervals=31, exclusions=exclusions,
    timezone='Europe/London', minimum_hours=MIN_VALID_HOURS))
print('Prepared the 23-home cohort and daily means.')

## Interpret the outputs

Review the saved CSV files and JSON checks in the selected output directory. Keep the settings in `analysis_settings.json`
fixed when reproducing this manuscript. If a source-data assertion fails, inspect the named records before changing
any thresholds. `README.md` documents the order of all manuscript stages.